In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings("ignore")




In [ ]:
import pandas as pd
import os

path="data/spy_minute_ohlcv.parquet"
data=pd.read_parquet(path)



In [6]:
def clean_minute_data(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Clean the minute-level OHLCV data.
      1. Retain only the trading hours from 09:30 to 16:00
      2. Handle missing values and abnormal volume
      3. Align timestamps
    '''
    df = df.copy()

    # --- 1. Index Check & Sorting ---
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='first')]

    # --- 2. Keep only the trading hours 09:30–16:00 ---
    # Note: The end time of the data is 15:59, so [09:30, 16:00) is equivalent to [09:30, 15:59]
    df = df.between_time('09:30', '15:59')

    # --- 3. Handling Outliers ---
    # volume / trades should not be negative
    df = df[(df['volume'] >= 0) & (df['trades'] >= 0)]

    # OHLC Consistency: high >= max(open, close), low <= min(open, close)
    bad_ohlc = (
        (df['high'] < df[['open', 'close']].max(axis=1)) |
        (df['low']  > df[['open', 'close']].min(axis=1)) |
        (df['high'] < df['low'])
    )
    if bad_ohlc.any():
        print(f"[clean] dropped {bad_ohlc.sum()} rows with inconsistent OHLC")
        df = df[~bad_ohlc]

    # --- 4. Missing Values ---
    # If the price column contains NaN values, drop the column; treat NaN values in the volume column as 0
    price_cols = ['open', 'high', 'low', 'close', 'vwap']
    n_before = len(df)
    df = df.dropna(subset=price_cols)
    if len(df) < n_before:
        print(f"[clean] dropped {n_before - len(df)} rows with NaN in price cols")
    df['volume'] = df['volume'].fillna(0)
    df['trades'] = df['trades'].fillna(0)

    return df

In [8]:
data=clean_minute_data(data)
print(f"shape: {data.shape}")
print(f"date range: {data.index.min()}  ->  {data.index.max()}")
print(f"unique trading days: {data.index.normalize().nunique()}")
data.head()

shape: (1078144, 7)
date range: 2015-01-02 09:30:00  ->  2025-12-31 15:59:00
unique trading days: 2766


,open,high,low,close,volume,trades,vwap
datetime,,,,,,,
2015-01-02 09:30:00,206.380005,206.449997,206.300003,206.389999,1658022,3555,206.385788
2015-01-02 09:31:00,206.389999,206.449997,206.289993,206.440002,449564,1936,206.361984
2015-01-02 09:32:00,206.440002,206.679993,206.430099,206.667999,723868,2794,206.567093
2015-01-02 09:33:00,206.669998,206.770004,206.320007,206.730103,641686,1586,206.548935
2015-01-02 09:34:00,206.740005,206.750000,206.600006,206.600006,315455,1450,206.681168


In [11]:
#一个交易日正常的bar数为390，我们去掉不满390个bar的交易日
bars_per_day=data.groupby(data.index.normalize()).size()
full_days=bars_per_day[bars_per_day==390].index
data=data[data.index.normalize().isin(full_days)]
os.makedirs("data", exist_ok=True)
data.to_parquet("data/spy_minute_clean.parquet")